# Train OLD-window policies — DDPG/TD3, Approach 1 & 2 (original recipe)

Retrains one online actor-critic policy with the **ORIGINAL pre-tuning recipe**:
the old (narrow) gain box and the original checkpoint selection
(`MAE + 0.15*overshoot` on the validation window — no undershoot term, no
full-day gate), exactly as the archived `policies/old_box/*_best_oldbox.pt`
were produced.

```
GAIN_LOW  = [-6.0, -0.100, -0.60]     # [Kp, Ki, Kw]  (old window)
GAIN_HIGH = [-0.05, -0.0001, 0.10]
```

**Usage:** set `ALGO` ('DDPG' or 'TD3') and `APPROACH` (1 or 2) in the setup cell,
then run top-to-bottom (~30-60 min). The result is saved to
`policies/old_box/<algo>_online_ac_approach<n>_best_oldbox.pt` — the tuned
`policies/*_best.pt` are never touched. With the default seed (42) this
reproduces the archived old-box policies (Juan-day MAE 0.109 for DDPG-A1/A2 and
TD3-A1, 0.183 for TD3-A2).


In [ ]:
# ── Setup: CHOOSE the run here ────────────────────────────────────────────────
ALGO     = 'DDPG'    # 'DDPG' or 'TD3'
APPROACH = 1         # 1 = search->freeze->critic-exploit, 2 = critic-driven

import os, sys, copy, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
sys.path.insert(0, os.getcwd())                                            # for: import config
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), os.pardir)))  # reach ../main_script
from main_script import *
import config as cfg
from config import *
configure(cfg); set_seed(SEED)

# ── OLD gain window (pinned explicitly; do NOT use the tuned wide box here) ──
cfg.GAIN_LOW  = np.array([-6.0, -0.100, -0.60], dtype=np.float32)
cfg.GAIN_HIGH = np.array([-0.05, -0.0001, 0.10], dtype=np.float32)

TWIN = (ALGO == 'TD3')
if APPROACH == 2:
    EPISODES = 600; UPDATES_PER_CYCLE = 2   # Approach-2: more critic-driven iterations
agent  = (TD3 if TWIN else DDPG)(warm_start=True)
buffer = ReplayBuffer()

TAG = f"{ALGO.lower()}_online_ac_approach{APPROACH}"
OB_DIR = os.path.join(SAVE_DIR, 'old_box'); os.makedirs(OB_DIR, exist_ok=True)
SAVE_PATH = os.path.join(OB_DIR, f'{TAG}_best_oldbox.pt')

# ── Datasets + episode sampler (Juan's new days) ──
train_datasets   = [load_dataset(f) for f in JUAN_FILES]
VAL_FILE         = JUAN_FILES[-1]
episode_datasets = [load_dataset(f) for f in JUAN_FILES if f != VAL_FILE]
train_eval_days  = [d for d in train_datasets if d["name"] != VAL_FILE]
val_eval_days    = [d for d in train_datasets if d["name"] == VAL_FILE]
def get_episode_data(datasets, ep_steps=SHORT_EP_STEPS):
    w = np.array([2.0 if "Cloudy" in d["name"] else 1.0 for d in datasets]); w = w / w.sum()
    idx = int(np.random.choice(len(datasets), p=w)); full = datasets[idx]
    ms = max(0, full["N"] - ep_steps - 1); start = np.random.randint(0, ms + 1) if ms > 0 else 0
    return window(full, start, ep_steps)
print(f"[{ALGO} A{APPROACH}] OLD-window training -> {os.path.relpath(SAVE_PATH, SAVE_DIR)}")
print(f"old box: LOW={list(cfg.GAIN_LOW)} HIGH={list(cfg.GAIN_HIGH)} | VAL={VAL_FILE}")


In [ ]:
# ── Initialise the critic FROM the trained CQL policy (same as the approach notebooks) ──
CQL_CKPT           = 'cql_cirl_actor_best.pt'
CQL_PRETRAIN_STEPS = 8000
CQL_PRETRAIN_NOISE = 0.03

def load_cql_actor(ckpt=CQL_CKPT):
    a = Actor().to(DEVICE)
    a.load_state_dict(torch.load(os.path.join(SAVE_DIR, ckpt), map_location=DEVICE, weights_only=True))
    a.eval(); print(f"[cql] policy loaded from {ckpt}")
    return a

def fill_buffer_from_actor(actor_net, datasets, buf, noise_std=CQL_PRETRAIN_NOISE):
    actor_net.eval()
    with torch.no_grad():
        for ds in datasets:
            env = SolarFieldEnv(ds); obs = env.reset(); done = False
            while not done:
                s = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
                a = actor_net(s).cpu().numpy()[0]
                a = np.clip(a + np.random.randn(3).astype(np.float32) * noise_std, 0.0, 1.0)
                obs2, r, done, info = env.step(a); buf.add(obs, a, r, obs2, done); obs = obs2
    return len(buf)

def pretrain_critic_from_cql(agent, cql_actor, datasets, n_updates=CQL_PRETRAIN_STEPS, batch=BATCH_SIZE):
    pre = ReplayBuffer(); n = fill_buffer_from_actor(cql_actor, datasets, pre)
    print(f"[cql-critic] {n} transitions from CQL rollouts; fitting critic for {n_updates} steps...")
    cql_actor.eval()
    for it in range(1, n_updates + 1):
        s, a, r, s2, d = pre.sample(batch)
        with torch.no_grad():
            a2 = cql_actor(s2)
            if TWIN:
                q1t, q2t = agent.critic_target(s2, a2); qt = r + agent.gamma * (1 - d) * torch.min(q1t, q2t)
            else:
                qt = r + agent.gamma * (1 - d) * agent.critic_target(s2, a2)
        if TWIN:
            q1, q2 = agent.critic(s, a); lc = F.mse_loss(q1, qt) + F.mse_loss(q2, qt)
        else:
            lc = F.mse_loss(agent.critic(s, a), qt)
        agent.opt_c.zero_grad(); lc.backward(); agent.opt_c.step()
        for tp, p in zip(agent.critic_target.parameters(), agent.critic.parameters()):
            tp.data.mul_(1 - agent.tau); tp.data.add_(agent.tau * p.data)
    print("[cql-critic] done; critic initialised as the CQL Q-function.")

cql_actor = load_cql_actor()
pretrain_critic_from_cql(agent, cql_actor, episode_datasets)


In [ ]:
# ── TRAIN (original pre-tuning recipe; branches on APPROACH) ─────────────────
import time
history = {'ep': [], 'train_reward': [], 'val_reward': [], 'train_mae': [], 'val_mae': [],
           'val_overshoot': [], 'Kp': [], 'Ki': [], 'Kw': []}
global_step = 0
noise = OUNoise(sigma=0.15, sigma_min=0.01, decay=0.999)
torch.save(agent.actor.state_dict(), SAVE_PATH)
_t0 = time.time()

if APPROACH == 1:
    # ── Approach 1: gain SEARCH -> FREEZE -> CRITIC-EXPLOIT (original code) ──
    EVAL_WIN       = 2000
    MAE_CAP_MARGIN = 1.05
    SEARCH_SIGMA0  = 0.45; SEARCH_SIGMA1 = 0.01
    SEARCH_GROW    = 1.5;  SEARCH_SHRINK = 0.6; SHRINK_PATIENCE = 3
    SEARCH_OVR_W   = 0.5
    SEARCH_PLATEAU_PCT = 0.0001; SEARCH_PLATEAU_PATIENCE = 60
    EARLY_STOP_PCT = 0.0001;     EARLY_STOP_PATIENCE = 50
    agent.bc_weight = BC_GUIDE_WEIGHT

    rng = np.random.default_rng(SEED)
    best_norm = normalize_gains(START_GAIN).astype(np.float32)
    bm, bo, best_rew = eval_gain(best_norm, train_eval_days)
    best_score = best_rew - SEARCH_OVR_W * bo
    search_sigma = SEARCH_SIGMA0; search_stall = 0
    agent.bc_target = torch.tensor(best_norm, dtype=torch.float32, device=DEVICE).view(1, -1)
    best_val_obj = float('inf'); best_overshoot = float('inf'); MAE_CAP = float('inf')
    best_exploit_rew = -1e9; stall = 0
    searching = True; search_ref = best_score; search_plateau = 0
    print(f"Start (expert): MAE={bm:.3f} overshoot={bo:.3f}\n")

    for ep in range(1, EPISODES + 1):
        froze_now = False
        if searching:
            if ep > 1:
                cand = np.clip(best_norm + rng.normal(0, search_sigma, 3).astype(np.float32), 0.0, 1.0)
                cm, co, cr = eval_gain(cand, train_eval_days)
                cscore = cr - SEARCH_OVR_W * co
                if cscore > best_score:
                    best_score, best_rew, best_norm = cscore, cr, cand
                    search_sigma = min(search_sigma * SEARCH_GROW, SEARCH_SIGMA0); search_stall = 0
                else:
                    search_stall += 1
                    if search_stall >= SHRINK_PATIENCE:
                        search_sigma = max(search_sigma * SEARCH_SHRINK, SEARCH_SIGMA1); search_stall = 0
                if best_score > search_ref + abs(search_ref) * SEARCH_PLATEAU_PCT:
                    search_ref = best_score; search_plateau = 0
                else:
                    search_plateau += 1
                froze_now = search_plateau >= SEARCH_PLATEAU_PATIENCE
            agent.bc_target = torch.tensor(best_norm, dtype=torch.float32, device=DEVICE).view(1, -1)
            agent.bc_alpha = 0.0; noise.reset(); use_noise = noise
        else:
            agent.bc_alpha = RL_FINE_ALPHA; use_noise = None

        data = get_episode_data(episode_datasets, SHORT_EP_STEPS); env = SolarFieldEnv(data)
        obs = env.reset(); done = False; gsum = np.zeros(3); steps = 0
        while not done:
            a = agent.act(obs, noise=use_noise)
            obs2, r, done, info = env.step(a); buffer.add(obs, a, r, obs2, done)
            gsum += denormalize_gains(a); global_step += 1
            if global_step >= WARMUP_STEPS and global_step % UPDATE_EVERY == 0:
                for _ in range(UPDATES_PER_CYCLE): agent.update(buffer, BATCH_SIZE)
            obs = obs2; steps += 1
            if use_noise is not None: noise.step()
        mg = gsum / max(steps, 1)

        tm, tr_, to = eval_actor(agent.actor, train_eval_days)
        vm, vr, vo  = eval_actor(agent.actor, val_eval_days)
        history['ep'].append(ep); history['train_reward'].append(tr_); history['val_reward'].append(vr)
        history['train_mae'].append(tm); history['val_mae'].append(vm); history['val_overshoot'].append(vo)
        history['Kp'].append(mg[0]); history['Ki'].append(mg[1]); history['Kw'].append(mg[2])

        tag = ''; vobj = vm + OVERSHOOT_WEIGHT * vo          # ORIGINAL selection (no undershoot)
        if searching and not froze_now:
            if vobj < best_val_obj:
                best_val_obj = vobj
                torch.save(agent.actor.state_dict(), SAVE_PATH); tag = '  <- saved (search)'
        elif froze_now:
            searching = False; MAE_CAP = vm * MAE_CAP_MARGIN; best_overshoot = vo
            torch.save(agent.actor.state_dict(), SAVE_PATH); tag = f'  <- FROZEN; MAE baseline {vm:.3f}'
        else:
            if vm <= MAE_CAP and vo < best_overshoot:
                best_overshoot = vo
                torch.save(agent.actor.state_dict(), SAVE_PATH); tag = '  <- lower overshoot (saved)'

        stop_now = False
        if not searching and not froze_now:
            if vr > best_exploit_rew + abs(best_exploit_rew) * EARLY_STOP_PCT:
                best_exploit_rew = vr; stall = 0
            else:
                stall += 1; stop_now = stall >= EARLY_STOP_PATIENCE
        if ep % EVAL_EVERY == 0 or ep == 1 or froze_now or stop_now:
            dg = denormalize_gains(best_norm); ph = 'SRCH' if searching else 'XPLT'
            print(f"Ep {ep:3d} [{ph}] sig={search_sigma:.3f} | Kp={dg[0]:+.3f} Ki={dg[1]:+.5f} "
                  f"Kw={dg[2]:+.4f} | valMAE={vm:.3f} ovr={vo:.3f}{tag}")
        if stop_now:
            print(f"EARLY STOP at ep {ep}"); break

else:
    # ── Approach 2: critic-driven (original code) ─────────────────────────────
    agent.bc_weight = 0.0
    best_val_obj = float('inf'); best_exploit_rew = -1e9; stall = 0
    EARLY_STOP_PCT, EARLY_STOP_PATIENCE = 0.0001, 100
    for ep in range(1, EPISODES + 1):
        data = get_episode_data(episode_datasets, SHORT_EP_STEPS); env = SolarFieldEnv(data)
        obs = env.reset(); done = False; gsum = np.zeros(3); steps = 0
        while not done:
            a = agent.act(obs, noise=noise)
            obs2, r, done, info = env.step(a); buffer.add(obs, a, r, obs2, done)
            gsum += denormalize_gains(a); global_step += 1
            if global_step >= WARMUP_STEPS and global_step % UPDATE_EVERY == 0:
                for _ in range(UPDATES_PER_CYCLE): agent.update(buffer, BATCH_SIZE)
            obs = obs2; steps += 1; noise.step()
        mg = gsum / max(steps, 1)

        tm, tr_, to = eval_actor(agent.actor, train_eval_days)
        vm, vr, vo  = eval_actor(agent.actor, val_eval_days)
        history['ep'].append(ep); history['train_reward'].append(tr_); history['val_reward'].append(vr)
        history['train_mae'].append(tm); history['val_mae'].append(vm); history['val_overshoot'].append(vo)
        history['Kp'].append(mg[0]); history['Ki'].append(mg[1]); history['Kw'].append(mg[2])

        tag = ''; vobj = vm + OVERSHOOT_WEIGHT * vo          # ORIGINAL selection (no undershoot, no gate)
        if vobj < best_val_obj:
            best_val_obj = vobj; torch.save(agent.actor.state_dict(), SAVE_PATH); tag = '  <- saved best'
        stop_now = False
        if vr > best_exploit_rew + abs(best_exploit_rew) * EARLY_STOP_PCT:
            best_exploit_rew = vr; stall = 0
        else:
            stall += 1; stop_now = stall >= EARLY_STOP_PATIENCE
        if ep % EVAL_EVERY == 0 or ep == 1 or stop_now:
            print(f"Ep {ep:3d} | Kp={mg[0]:+.3f} Ki={mg[1]:+.5f} Kw={mg[2]:+.4f} "
                  f"| valMAE={vm:.3f} ovr={vo:.3f}{tag}")
        if stop_now:
            print(f"EARLY STOP at ep {ep}: val reward plateaued."); break

print(f"\nDone in {(time.time()-_t0)/60:.1f} min. Saved -> {SAVE_PATH}")


In [ ]:
# ── Final evaluation on Juan's 4 days (old-window policy just trained) ────────
def undershoot_metric(T, tref):
    above = np.nonzero(T >= tref)[0]
    return float(np.max(tref - T[above[0]:])) if len(above) else float(np.max(tref - T))

best_actor = Actor().to(DEVICE)
best_actor.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE, weights_only=True)); best_actor.eval()

print(f"{'Juan day':40s} | {TAG} MAE   RMSE   ovr    und | Expert MAE   ovr    und")
print('-' * 108)
rows = []
for ds in train_datasets:
    tr = dataset_tref(ds['name'])
    T, Q, G   = rollout_full(best_actor, ds)
    Te, Qe    = rollout_expert(ds)
    m, rm     = mae_rmse(T, tr);  ov  = peak_overshoot(T, tr);  un  = undershoot_metric(T, tr)
    me, rme   = mae_rmse(Te, tr); ove = peak_overshoot(Te, tr); une = undershoot_metric(Te, tr)
    rows.append((ds['name'], m, rm, ov, un, me, ove, une))
    print(f"{ds['name']:40s} | {m:6.3f} {rm:6.3f} {ov:6.3f} {un:6.3f} | {me:6.3f} {ove:6.3f} {une:6.3f}")
print('-' * 108)
print("MEAN over Juan days:  %s MAE=%.3f RMSE=%.3f ovr=%.3f und=%.3f  |  Expert MAE=%.3f ovr=%.3f und=%.3f"
      % (TAG, np.mean([r[1] for r in rows]), np.mean([r[2] for r in rows]), np.mean([r[3] for r in rows]),
         np.mean([r[4] for r in rows]),
         np.mean([r[5] for r in rows]), np.mean([r[6] for r in rows]), np.mean([r[7] for r in rows])))
